# 🤖 Agentic RAG — Full Pipeline

This notebook implements a full Agentic RAG pipeline with all 6 components:

| # | Block | Color | Role |
|---|-------|-------|------|
| 1 | **Query Planning & Decomposition** | 🟡 Yellow | Breaks complex queries into focused sub-queries |
| 2 | **Chain of Thought** | 🩷 Pink | Forces step-by-step reasoning before answering |
| 3 | **ReAct Agent** | 🔴 Red | Decides which tool to call (FAISS / arXiv / Wikipedia) |
| 4 | **Iterative Retrieval Check** | 🟢 Green | Validates doc relevance *before* generating |
| 5 | **Answer Synthesis** | 🔵 Blue | Merges multi-source answers into one coherent response |
| 6 | **Self Reflection** | 🟠 Orange | LLM judges its own answer and loops if needed |

**Retrieval Priority:** Local FAISS vector store → arXiv → Wikipedia

**Chunking Strategy:** Hybrid — RecursiveCharacterTextSplitter (semantic) + fixed-size overlapping chunks merged via MMR deduplication

---
## Cell 1 — Imports

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 │ Imports
# All third-party dependencies are imported here so missing packages surface
# immediately before any work begins.
# ─────────────────────────────────────────────────────────────────────────────

import logging
import operator
import os
import pickle
from pathlib import Path
from typing import Annotated, List, Literal, Optional

import arxiv                                          # pip install arxiv
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph

# ── Cell health-check ────────────────────────────────────────────────────────
# Verifies all imports resolved correctly before continuing.
print("✅ Cell 1 passed — all imports resolved.")

---
## Cell 2 — Logging & Config

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 │ Logging & Config
#
# Logging writes to BOTH the notebook console and a local file so that:
#   - Developers see real-time progress in the notebook output
#   - A permanent record is saved to disk for post-run debugging
#
# All tuneable parameters live here so developers never need to hunt
# through node functions to change settings.
# ─────────────────────────────────────────────────────────────────────────────

# ── Logging setup ────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    handlers=[
        logging.FileHandler("agentic_rag.log"),   # persisted to disk
        logging.StreamHandler(),                   # visible in notebook output
    ],
)
# Silence noisy third-party HTTP logs — only show warnings and above
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("faiss").setLevel(logging.WARNING)

logger = logging.getLogger("agentic_rag")
logger.info("Logging initialised → writing to 'agentic_rag.log'")

# ── Environment ──────────────────────────────────────────────────────────────
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

# ── Tuneable parameters ───────────────────────────────────────────────────────
# Paths
PDF_FOLDER        = Path("./pdf")            # folder containing source PDFs
VECTORSTORE_PATH  = Path("./faiss_index")    # where the FAISS index is saved/loaded

# Hybrid chunking
RECURSIVE_CHUNK_SIZE    = 1000   # characters — semantic-boundary splitter
RECURSIVE_CHUNK_OVERLAP = 150    # overlap keeps context across chunk boundaries
FIXED_CHUNK_SIZE        = 400    # characters — fine-grained fixed-size splitter
FIXED_CHUNK_OVERLAP     = 50

# Retrieval
TOP_K_RETRIEVAL    = 6    # how many docs to fetch from FAISS per sub-query
RELEVANCE_THRESHOLD = 0.30  # cosine similarity floor — docs below this are discarded

# Agent loop
MAX_ATTEMPTS       = 3    # maximum self-reflection retry cycles

# Models
LLM_MODEL          = "openai:gpt-4o"
EMBEDDING_MODEL    = "text-embedding-3-small"

# ── LLM & Embeddings ─────────────────────────────────────────────────────────
llm        = init_chat_model(LLM_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
logger.info(f"LLM: {LLM_MODEL} | Embeddings: {EMBEDDING_MODEL}")

# ── Cell health-check ────────────────────────────────────────────────────────
assert os.environ.get("OPENAI_API_KEY"), "❌ OPENAI_API_KEY not found — check your .env file."
assert PDF_FOLDER.exists(), f"❌ PDF folder not found at '{PDF_FOLDER.resolve()}'"
print("✅ Cell 2 passed — config loaded, API key found, PDF folder exists.")

---
## Cell 3 — Hybrid Chunking & Vector Store
> Loads or builds the FAISS index. If an existing index is found on disk it is reused — no re-embedding needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 │ Hybrid Chunking & Vector Store
#
# HYBRID CHUNKING STRATEGY
# ─────────────────────────
# Using two complementary splitters produces richer retrieval coverage:
#
#  1. RecursiveCharacterTextSplitter (large chunks, 1000 chars)
#     Splits on paragraph/sentence boundaries first, so each chunk tends to
#     preserve a complete semantic idea.  Good for open-ended questions.
#
#  2. Fixed-size overlapping splitter (small chunks, 400 chars)
#     Ignores boundaries and cuts at fixed widths with heavy overlap.
#     Captures fine-grained facts that large chunks might bury.
#
# Both chunk sets are merged into a single FAISS index.  At query time,
# MMR (Maximal Marginal Relevance) is used so the retriever returns
# diverse, non-redundant chunks instead of near-duplicates.
#
# VECTOR STORE REUSE
# ───────────────────
# If FAISS_INDEX_PATH already exists on disk the index is loaded directly,
# skipping PDF loading, splitting and embedding — saving time and API cost.
# ─────────────────────────────────────────────────────────────────────────────

def load_pdfs(folder: Path) -> List[Document]:
    """
    Recursively loads all PDF files from *folder*.
    Returns a flat list of LangChain Document objects (one per page).
    Logs a warning for any file that fails to load rather than crashing.
    """
    all_docs: List[Document] = []
    pdf_files = list(folder.glob("**/*.pdf"))
    logger.info(f"[load_pdfs] Found {len(pdf_files)} PDF(s) in '{folder}'")

    for pdf_path in pdf_files:
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
            all_docs.extend(pages)
            logger.info(f"[load_pdfs] ✅ Loaded {len(pages)} pages from '{pdf_path.name}'")
        except Exception as exc:
            logger.warning(f"[load_pdfs] ⚠️  Skipped '{pdf_path.name}': {exc}")

    return all_docs


def hybrid_chunk(docs: List[Document]) -> List[Document]:
    """
    Applies two splitting strategies and returns the merged chunk list.

    Pass 1 — RecursiveCharacterTextSplitter
        Respects paragraph / sentence boundaries.  Produces large,
        semantically coherent chunks suited for open-ended Q&A.

    Pass 2 — Fixed-size splitter (same class, no separator preference)
        Produces small, densely overlapping chunks suited for
        precise fact retrieval.

    Duplicates across the two passes are acceptable — MMR in the
    retriever handles diversity at query time.
    """
    # Pass 1: semantic-boundary chunks
    semantic_splitter = RecursiveCharacterTextSplitter(
        chunk_size=RECURSIVE_CHUNK_SIZE,
        chunk_overlap=RECURSIVE_CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],  # prefer paragraph > sentence > word
    )
    semantic_chunks = semantic_splitter.split_documents(docs)
    logger.info(f"[hybrid_chunk] Pass 1 (semantic): {len(semantic_chunks)} chunks")

    # Pass 2: fine-grained fixed-size chunks
    fixed_splitter = RecursiveCharacterTextSplitter(
        chunk_size=FIXED_CHUNK_SIZE,
        chunk_overlap=FIXED_CHUNK_OVERLAP,
        separators=[""],  # no separator preference — purely size-based
    )
    fixed_chunks = fixed_splitter.split_documents(docs)
    logger.info(f"[hybrid_chunk] Pass 2 (fixed):    {len(fixed_chunks)} chunks")

    combined = semantic_chunks + fixed_chunks
    logger.info(f"[hybrid_chunk] Total combined:    {len(combined)} chunks")
    return combined


def build_or_load_vectorstore() -> FAISS:
    """
    Returns a FAISS vector store, either by:
      - Loading an existing index from VECTORSTORE_PATH (fast, no API calls), or
      - Building a new one from PDFs in PDF_FOLDER and saving it to disk.

    Saving the index locally avoids re-embedding on every notebook restart.
    """
    if VECTORSTORE_PATH.exists():
        logger.info(f"[vectorstore] ♻️  Loading existing FAISS index from '{VECTORSTORE_PATH}'")
        store = FAISS.load_local(
            str(VECTORSTORE_PATH),
            embeddings,
            allow_dangerous_deserialization=True,   # safe — we wrote this file ourselves
        )
        logger.info("[vectorstore] ✅ Existing index loaded successfully")
        return store

    # No existing index — build from scratch
    logger.info("[vectorstore] 🔨 No existing index found — building from PDFs...")
    raw_docs = load_pdfs(PDF_FOLDER)

    if not raw_docs:
        raise ValueError(f"No documents loaded from '{PDF_FOLDER}'. Add PDFs and retry.")

    chunks = hybrid_chunk(raw_docs)

    if not chunks:
        raise ValueError("Chunking produced 0 chunks — check PDF content.")

    logger.info("[vectorstore] 🔗 Embedding chunks (this may take a moment)...")
    store = FAISS.from_documents(chunks, embeddings)

    # Persist to disk for future reuse
    VECTORSTORE_PATH.mkdir(parents=True, exist_ok=True)
    store.save_local(str(VECTORSTORE_PATH))
    logger.info(f"[vectorstore] 💾 Index saved to '{VECTORSTORE_PATH}'")

    return store


# ── Build / load the vector store ────────────────────────────────────────────
vectorstore = build_or_load_vectorstore()

# MMR retriever — fetches TOP_K_RETRIEVAL docs but maximises diversity
# fetch_k > k so MMR has a pool to select from
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K_RETRIEVAL, "fetch_k": TOP_K_RETRIEVAL * 3},
)
logger.info(f"[vectorstore] Retriever ready (MMR, k={TOP_K_RETRIEVAL})")

# ── Cell health-check ────────────────────────────────────────────────────────
assert vectorstore is not None, "❌ Vector store failed to initialise."
assert retriever   is not None, "❌ Retriever failed to initialise."
print("✅ Cell 3 passed — vector store and retriever ready.")

---
## Cell 4 — External Tools (arXiv & Wikipedia)
> These are **fallback** sources only — the FAISS vector store is always tried first.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 │ External Tools — arXiv & Wikipedia
#
# These tools are ONLY invoked when the FAISS vector store does not return
# sufficiently relevant documents (controlled by RELEVANCE_THRESHOLD in
# the Iterative Retrieval Check node).
#
# Priority order enforced by the ReAct agent (Cell 6):
#   1. FAISS (local, fast, authoritative)
#   2. arXiv (academic papers — good for technical / research questions)
#   3. Wikipedia (general background knowledge)
# ─────────────────────────────────────────────────────────────────────────────

def search_arxiv(query: str, max_results: int = 3) -> List[Document]:
    """
    Searches arXiv for papers matching *query*.
    Returns a list of Documents where page_content is the paper abstract
    and metadata contains title, authors, and PDF URL.

    Called by the ReAct agent when FAISS retrieval is insufficient.
    """
    logger.info(f"[arxiv] 🔍 Searching arXiv: '{query}'")
    try:
        client  = arxiv.Client()
        search  = arxiv.Search(query=query, max_results=max_results)
        results = list(client.results(search))

        docs = [
            Document(
                page_content=r.summary,
                metadata={
                    "source": "arxiv",
                    "title": r.title,
                    "authors": ", ".join(a.name for a in r.authors[:3]),
                    "url": r.pdf_url,
                },
            )
            for r in results
        ]
        logger.info(f"[arxiv] ✅ Retrieved {len(docs)} paper(s)")
        return docs

    except Exception as exc:
        logger.error(f"[arxiv] ❌ Search failed: {exc}")
        return []


def search_wikipedia(query: str) -> List[Document]:
    """
    Searches Wikipedia for *query* and returns the top result as a Document.
    Called by the ReAct agent as a last-resort fallback after FAISS and arXiv.
    """
    logger.info(f"[wikipedia] 🔍 Searching Wikipedia: '{query}'")
    try:
        wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=2))
        result    = wiki_tool.run(query)

        doc = Document(
            page_content=result,
            metadata={"source": "wikipedia", "query": query},
        )
        logger.info("[wikipedia] ✅ Retrieved Wikipedia content")
        return [doc]

    except Exception as exc:
        logger.error(f"[wikipedia] ❌ Search failed: {exc}")
        return []


# ── Cell health-check ────────────────────────────────────────────────────────
# Smoke-test: confirm the functions are callable (no live API calls here)
assert callable(search_arxiv),     "❌ search_arxiv is not callable"
assert callable(search_wikipedia), "❌ search_wikipedia is not callable"
print("✅ Cell 4 passed — external tool functions defined.")

---
## Cell 5 — State & Prompts

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 │ State Schema & Prompts
#
# AgenticRAGState
# ───────────────
# The single source of truth passed between every node.  Each node reads
# relevant fields and returns state.model_copy(update={...}) with its changes.
# Using model_copy() instead of mutation keeps state immutable per LangGraph
# best-practice and makes debugging straightforward.
#
# Prompt Design Notes
# ────────────────────
# • decomposition_prompt  — instructs the LLM to output JSON, making it easy
#                           to parse sub-queries reliably.
# • cot_prompt            — includes a scratchpad section so reasoning is
#                           explicit and auditable in the logs.
# • reflection_prompt     — receives original_question (not the rewritten
#                           query) so judgment is always against user intent.
# ─────────────────────────────────────────────────────────────────────────────

# ── State Schema ─────────────────────────────────────────────────────────────

class AgenticRAGState(BaseModel):
    # ── Input ────────────────────────────────────────────────────────────────
    original_question: str              # never mutated — used by reflector
    question: str                       # current active query (may be rewritten)

    # ── Planning (Block 1) ───────────────────────────────────────────────────
    sub_queries: List[str] = []         # decomposed sub-questions

    # ── Retrieval (Blocks 3 & 4) ─────────────────────────────────────────────
    retrieved_docs: List[Document] = [] # current set of retrieved docs (replaced each loop)
    retrieval_source: str = ""          # which source was used: faiss / arxiv / wikipedia
    docs_are_relevant: bool = False     # set by iterative retrieval check node

    # ── Reasoning (Block 2) ──────────────────────────────────────────────────
    cot_reasoning: str = ""             # chain-of-thought scratchpad

    # ── Answer (Blocks 5 & 6) ────────────────────────────────────────────────
    sub_answers: List[str] = []         # one answer per sub-query
    final_answer: str = ""              # synthesised answer
    reflection: str = ""                # reflection verdict
    needs_revision: bool = False        # True → loop back for another attempt

    # ── Loop control ─────────────────────────────────────────────────────────
    attempts: int = 0


# ── Prompts ───────────────────────────────────────────────────────────────────

# 1. Query Decomposition — returns JSON list of sub-queries
decomposition_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query planning assistant. "
     "Break the user question into 1–4 focused sub-questions that together fully cover the original question. "
     "If the question is already simple and focused, return it as a single-item list. "
     "Respond ONLY with a valid JSON array of strings. Example: [\"sub-q 1\", \"sub-q 2\"]"),
    ("human", "Question: {question}"),
])

# 2. Chain of Thought — forces explicit step-by-step reasoning
cot_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful analytical assistant. "
     "Before writing your answer, work through your reasoning step by step in a <scratchpad> section. "
     "Only use information from the provided context — do not invent facts. "
     "After your scratchpad, write a concise final answer."),
    ("human",
     "Context:\n{context}\n\n"
     "Sub-question: {sub_question}\n\n"
     "<scratchpad>\n(reason step by step here)\n</scratchpad>\n\n"
     "Answer:"),
])

# 3. ReAct tool-selection — chooses between faiss / arxiv / wikipedia
react_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a retrieval routing agent. "
     "Given the query and the relevance verdict of the last FAISS retrieval, "
     "decide which tool to use next. "
     "Rules: "
     "1. Always prefer 'faiss' if FAISS docs are relevant. "
     "2. Use 'arxiv' for academic, technical, or research topics when FAISS fails. "
     "3. Use 'wikipedia' only as a last resort for general background. "
     "Respond with exactly one word: faiss, arxiv, or wikipedia."),
    ("human",
     "Query: {query}\n"
     "FAISS docs relevant: {faiss_relevant}\n"
     "Previous source used: {previous_source}"),
])

# 4. Document relevance check — scores whether docs match the query
relevance_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a relevance evaluator. "
     "Given a query and a set of retrieved document excerpts, decide if the documents "
     "contain enough relevant information to answer the query. "
     "Respond ONLY with YES or NO."),
    ("human",
     "Query: {query}\n\n"
     "Documents:\n{docs_preview}"),
])

# 5. Answer Synthesis — merges multiple sub-answers into one coherent response
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an answer synthesis expert. "
     "You will receive the original question and a set of partial answers, each addressing a sub-question. "
     "Synthesise them into one clear, complete, and non-repetitive final answer. "
     "If any partial answers contradict each other, note the discrepancy."),
    ("human",
     "Original question: {original_question}\n\n"
     "Partial answers:\n{sub_answers}"),
])

# 6. Self Reflection — judges if the final answer is complete
reflection_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a critical answer evaluator. "
     "Judge whether the answer completely and accurately addresses the original question "
     "based only on the provided context. Do NOT question whether the data is real. "
     "Respond strictly in this format:\n"
     "Reflection: YES or NO\n"
     "Explanation: <one sentence>"),
    ("human",
     "Context:\n{context}\n\n"
     "Original question: {original_question}\n\n"
     "Answer: {final_answer}"),
])

# ── Cell health-check ────────────────────────────────────────────────────────
assert AgenticRAGState.__fields__, "❌ AgenticRAGState has no fields"
assert decomposition_prompt and cot_prompt and react_prompt, "❌ One or more prompts not defined"
assert synthesis_prompt and reflection_prompt and relevance_prompt, "❌ One or more prompts not defined"
print("✅ Cell 5 passed — state schema and all 6 prompts defined.")

---
## Cell 6 — Node Functions (All 6 Blocks)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 │ Node Functions
#
# Each function corresponds to one block in the diagram:
#   query_planner        → Block 1 (Yellow)  Query Planning & Decomposition
#   cot_answerer         → Block 2 (Pink)    Chain of Thought
#   react_router         → Block 3 (Red)     ReAct Tool Selection
#   iterative_retrieval  → Block 4 (Green)   Iterative Retrieval Check
#   answer_synthesiser   → Block 5 (Blue)    Answer Synthesis
#   self_reflector       → Block 6 (Orange)  Self Reflection
#
# Helper nodes:
#   faiss_retriever      → fetches docs from the local FAISS index
#   query_rewriter       → rewrites the query when reflection says NO
#   should_continue      → routing function for the conditional edge
# ─────────────────────────────────────────────────────────────────────────────

import json
import re

# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 1 — Query Planning & Decomposition (Yellow)
# ─────────────────────────────────────────────────────────────────────────────

def query_planner(state: AgenticRAGState) -> AgenticRAGState:
    """
    Decomposes *state.question* into a list of focused sub-queries.

    Why: A single broad question often retrieves vague results. Breaking it
    into narrow sub-questions allows each to pull precise, targeted chunks.

    Output: state.sub_queries — list of strings, minimum 1 item.
    """
    logger.info(f"[query_planner] Decomposing: '{state.question}'")
    try:
        chain  = decomposition_prompt | llm | StrOutputParser()
        raw    = chain.invoke({"question": state.question})
        # Strip markdown fences if model wraps output in ```json ... ```
        clean  = re.sub(r"```(?:json)?|```", "", raw).strip()
        parsed = json.loads(clean)
        sub_queries = [q.strip() for q in parsed if isinstance(q, str) and q.strip()]
        logger.info(f"[query_planner] {len(sub_queries)} sub-quer(ies): {sub_queries}")
    except Exception as exc:
        logger.error(f"[query_planner] Decomposition failed ({exc}) — using original question")
        sub_queries = [state.question]

    return state.model_copy(update={"sub_queries": sub_queries})


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 3 — ReAct Tool Selection (Red)
# Placed before Block 2 because retrieval must happen before CoT answering.
# ─────────────────────────────────────────────────────────────────────────────

def faiss_retriever(state: AgenticRAGState) -> AgenticRAGState:
    """
    Retrieves documents from the local FAISS vector store.
    This is always the FIRST retrieval attempt — external sources are
    only considered if this node returns low-relevance results.

    Uses the first sub-query (or the full question if no decomposition).
    """
    query = state.sub_queries[0] if state.sub_queries else state.question
    logger.info(f"[faiss_retriever] Querying FAISS: '{query}'")
    try:
        docs = retriever.invoke(query)
        logger.info(f"[faiss_retriever] ✅ Retrieved {len(docs)} doc(s)")
    except Exception as exc:
        logger.error(f"[faiss_retriever] ❌ Retrieval failed: {exc}")
        docs = []

    return state.model_copy(update={
        "retrieved_docs":   docs,
        "retrieval_source": "faiss",
    })


def react_router(state: AgenticRAGState) -> AgenticRAGState:
    """
    ReAct routing: Reason → Act → Observe

    The LLM reasons about whether the FAISS docs are sufficient and decides
    which tool to invoke next (faiss / arxiv / wikipedia).  If a non-FAISS
    source is selected, the relevant search function is called and the docs
    are merged with any existing FAISS results.

    Priority enforced: faiss > arxiv > wikipedia
    """
    query = state.sub_queries[0] if state.sub_queries else state.question
    logger.info(f"[react_router] Deciding tool for: '{query}'")

    try:
        chain  = react_prompt | llm | StrOutputParser()
        choice = chain.invoke({
            "query":           query,
            "faiss_relevant":  str(state.docs_are_relevant),
            "previous_source": state.retrieval_source,
        }).strip().lower()
        logger.info(f"[react_router] Tool selected: '{choice}'")
    except Exception as exc:
        logger.error(f"[react_router] Routing failed ({exc}) — defaulting to faiss")
        choice = "faiss"

    new_docs = state.retrieved_docs  # start with existing docs

    if choice == "arxiv":
        arxiv_docs = search_arxiv(query)
        new_docs   = arxiv_docs + state.retrieved_docs  # arxiv results first
        source     = "arxiv"
    elif choice == "wikipedia":
        wiki_docs = search_wikipedia(query)
        new_docs  = wiki_docs + state.retrieved_docs
        source    = "wikipedia"
    else:
        # Stay with FAISS — no external call needed
        source = "faiss"

    return state.model_copy(update={
        "retrieved_docs":   new_docs,
        "retrieval_source": source,
    })


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 4 — Iterative Retrieval Check (Green)
# ─────────────────────────────────────────────────────────────────────────────

def iterative_retrieval_check(state: AgenticRAGState) -> AgenticRAGState:
    """
    Evaluates whether the retrieved documents are relevant enough to
    answer the current sub-query BEFORE spending tokens on generation.

    Two-stage check:
      1. Fast cosine similarity check — if any doc exceeds RELEVANCE_THRESHOLD
         it is considered relevant without an LLM call (saves cost).
      2. LLM relevance judge — used when all similarity scores are borderline.

    Sets state.docs_are_relevant = True/False for downstream routing.
    """
    query = state.sub_queries[0] if state.sub_queries else state.question
    logger.info(f"[iterative_check] Checking {len(state.retrieved_docs)} doc(s) for relevance")

    if not state.retrieved_docs:
        logger.warning("[iterative_check] No docs to evaluate → not relevant")
        return state.model_copy(update={"docs_are_relevant": False})

    # Stage 1: cosine similarity via FAISS score (fast, no LLM call)
    try:
        scored_docs = vectorstore.similarity_search_with_score(query, k=3)
        best_score  = scored_docs[0][1] if scored_docs else 0.0
        # FAISS returns L2 distance — convert to similarity (lower = better)
        # A threshold of 0.30 on L2 distance works well for text-embedding-3-small
        if best_score < RELEVANCE_THRESHOLD:
            logger.info(f"[iterative_check] ✅ Fast check passed (score={best_score:.3f})")
            return state.model_copy(update={"docs_are_relevant": True})
    except Exception as exc:
        logger.warning(f"[iterative_check] Similarity check error: {exc} — falling back to LLM")

    # Stage 2: LLM relevance judge (slower but more nuanced)
    docs_preview = "\n---\n".join(d.page_content[:300] for d in state.retrieved_docs[:4])
    try:
        chain  = relevance_prompt | llm | StrOutputParser()
        result = chain.invoke({"query": query, "docs_preview": docs_preview})
        is_relevant = "yes" in result.strip().lower()
        logger.info(f"[iterative_check] LLM verdict: {'RELEVANT' if is_relevant else 'NOT RELEVANT'}")
    except Exception as exc:
        logger.error(f"[iterative_check] LLM check failed: {exc} — assuming relevant")
        is_relevant = True  # fail-safe: don't stall the pipeline

    return state.model_copy(update={"docs_are_relevant": is_relevant})


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 2 — Chain of Thought Answering (Pink)
# ─────────────────────────────────────────────────────────────────────────────

def cot_answerer(state: AgenticRAGState) -> AgenticRAGState:
    """
    Iterates over each sub-query and generates a CoT answer for each one.

    Chain of Thought forces the LLM to write explicit step-by-step reasoning
    in a <scratchpad> before committing to an answer — this reduces
    hallucination and makes the reasoning auditable in state.cot_reasoning.

    Each sub-query reuses the same retrieved_docs pool.  For a more advanced
    setup, each sub-query could trigger its own retrieval pass.
    """
    logger.info(f"[cot_answerer] Answering {len(state.sub_queries)} sub-quer(ies) (attempt {state.attempts + 1})")
    context     = "\n\n".join(d.page_content for d in state.retrieved_docs)
    sub_answers = []
    reasoning   = []

    for i, sub_q in enumerate(state.sub_queries):
        try:
            chain  = cot_prompt | llm | StrOutputParser()
            result = chain.invoke({"context": context, "sub_question": sub_q})
            # Extract scratchpad for logging and the final answer
            if "</scratchpad>" in result:
                parts     = result.split("</scratchpad>")
                scratchpad = parts[0].replace("<scratchpad>", "").strip()
                answer     = parts[1].strip()
            else:
                scratchpad = ""
                answer     = result.strip()

            reasoning.append(f"Sub-Q {i+1}: {scratchpad}")
            sub_answers.append(f"Q: {sub_q}\nA: {answer}")
            logger.info(f"[cot_answerer] Sub-Q {i+1}/{len(state.sub_queries)} answered")

        except Exception as exc:
            logger.error(f"[cot_answerer] Failed on sub-Q {i+1}: {exc}")
            sub_answers.append(f"Q: {sub_q}\nA: [Error generating answer]")

    return state.model_copy(update={
        "sub_answers":   sub_answers,
        "cot_reasoning": "\n\n".join(reasoning),
        "attempts":      state.attempts + 1,
    })


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 5 — Answer Synthesis (Blue)
# ─────────────────────────────────────────────────────────────────────────────

def answer_synthesiser(state: AgenticRAGState) -> AgenticRAGState:
    """
    Merges all sub-answers into a single coherent final answer.

    If only one sub-query was planned the synthesis step is lightweight —
    it still runs for consistency (normalising formatting, removing
    redundancy) but does minimal work.

    Sources used are appended to the answer for transparency.
    """
    logger.info(f"[synthesiser] Synthesising {len(state.sub_answers)} sub-answer(s)")
    try:
        chain  = synthesis_prompt | llm | StrOutputParser()
        merged = chain.invoke({
            "original_question": state.original_question,
            "sub_answers":       "\n\n".join(state.sub_answers),
        })
        # Append source attribution
        source_note = f"\n\n📚 Sources used: {state.retrieval_source.upper()}"
        final       = merged.strip() + source_note
        logger.info("[synthesiser] ✅ Final answer synthesised")
    except Exception as exc:
        logger.error(f"[synthesiser] Synthesis failed: {exc}")
        final = "\n\n".join(state.sub_answers)  # fall back to concatenation

    return state.model_copy(update={"final_answer": final})


# ─────────────────────────────────────────────────────────────────────────────
# BLOCK 6 — Self Reflection (Orange)
# ─────────────────────────────────────────────────────────────────────────────

def self_reflector(state: AgenticRAGState) -> AgenticRAGState:
    """
    Evaluates the final answer against the ORIGINAL question (not the
    rewritten query) so that judgment always reflects user intent.

    Sets state.needs_revision = True if the answer is incomplete, which
    causes the graph to loop back through query_rewriter → retrieval.
    Loops are capped by MAX_ATTEMPTS to prevent infinite cycles.
    """
    logger.info(f"[reflector] Reflecting on answer (attempt {state.attempts})")
    context = "\n\n".join(d.page_content for d in state.retrieved_docs)

    try:
        chain  = reflection_prompt | llm | StrOutputParser()
        result = chain.invoke({
            "context":           context,
            "original_question": state.original_question,
            "final_answer":      state.final_answer,
        })
        is_good = "reflection: yes" in result.lower()
        logger.info(f"[reflector] {'✅ APPROVED' if is_good else '🔁 NEEDS REVISION'}")
    except Exception as exc:
        logger.error(f"[reflector] Reflection failed: {exc} — approving to avoid loop")
        result  = "Error during reflection."
        is_good = True

    return state.model_copy(update={
        "reflection":     result,
        "needs_revision": not is_good,
    })


# ─────────────────────────────────────────────────────────────────────────────
# Helper — Query Rewriter (used when reflection says NO)
# ─────────────────────────────────────────────────────────────────────────────

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a search query optimiser. "
     "Given the original question and why the previous answer was insufficient, "
     "rewrite the query to be more specific and likely to retrieve better documents. "
     "Return ONLY the rewritten query, no explanation."),
    ("human",
     "Original question: {question}\n"
     "Why it failed: {reflection}\n\n"
     "Rewritten query:"),
])

def query_rewriter(state: AgenticRAGState) -> AgenticRAGState:
    """
    Rewrites state.question based on the reflection feedback so that the
    next retrieval pass searches for something meaningfully different.
    Falls back to the original question if the rewrite fails.
    """
    logger.info("[query_rewriter] Rewriting query based on reflection")
    try:
        chain        = rewrite_prompt | llm | StrOutputParser()
        new_question = chain.invoke({
            "question":   state.original_question,
            "reflection": state.reflection,
        }).strip()
        logger.info(f"[query_rewriter] New query: '{new_question}'")
    except Exception as exc:
        logger.error(f"[query_rewriter] Rewrite failed: {exc} — keeping original")
        new_question = state.original_question

    return state.model_copy(update={
        "question":   new_question,
        "sub_queries": [],   # reset so query_planner re-decomposes the new question
    })


# ─────────────────────────────────────────────────────────────────────────────
# Routing function — controls the self-reflection loop exit condition
# ─────────────────────────────────────────────────────────────────────────────

def should_continue(state: AgenticRAGState) -> str:
    """
    Returns 'query_rewriter' if the answer needs revision and attempts remain.
    Returns END otherwise.
    """
    if not state.needs_revision:
        logger.info("[router] Answer approved → END")
        return END
    if state.attempts >= MAX_ATTEMPTS:
        logger.warning(f"[router] Max attempts ({MAX_ATTEMPTS}) reached → END")
        return END
    logger.info(f"[router] Needs revision (attempt {state.attempts}) → query_rewriter")
    return "query_rewriter"


def should_call_react(state: AgenticRAGState) -> str:
    """
    After the iterative relevance check:
      - If docs are relevant → proceed directly to CoT answering
      - If not relevant     → run ReAct to select a different source
    """
    if state.docs_are_relevant:
        logger.info("[router] Docs relevant → cot_answerer")
        return "cot_answerer"
    logger.info("[router] Docs not relevant → react_router")
    return "react_router"


# ── Cell health-check ────────────────────────────────────────────────────────
node_fns = [query_planner, faiss_retriever, react_router, iterative_retrieval_check,
            cot_answerer, answer_synthesiser, self_reflector, query_rewriter]
assert all(callable(fn) for fn in node_fns), "❌ One or more node functions are not callable"
print("✅ Cell 6 passed — all node functions defined.")

---
## Cell 7 — LangGraph DAG

```
START
  │
  ▼
query_planner (Block 1)
  │
  ▼
faiss_retriever
  │
  ▼
iterative_check (Block 4)
  │
  ├─ docs relevant ──────────────────────────────┐
  │                                              ▼
  └─ not relevant ──► react_router (Block 3) ──► cot_answerer (Block 2)
                                                  │
                                                  ▼
                                          answer_synthesiser (Block 5)
                                                  │
                                                  ▼
                                          self_reflector (Block 6)
                                                  │
                                    ┌─────────────┴──────────────┐
                                  END                     query_rewriter
                                                                  │
                                                           query_planner (loop)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 │ LangGraph DAG
#
# Graph wiring mirrors the 6-block diagram.
# path_map is provided on every conditional edge so LangGraph can render
# the full graph topology even before execution.
# ─────────────────────────────────────────────────────────────────────────────

builder = StateGraph(AgenticRAGState)

# ── Register nodes ───────────────────────────────────────────────────────────
builder.add_node("query_planner",        query_planner)
builder.add_node("faiss_retriever",      faiss_retriever)
builder.add_node("iterative_check",      iterative_retrieval_check)
builder.add_node("react_router",         react_router)
builder.add_node("cot_answerer",         cot_answerer)
builder.add_node("answer_synthesiser",   answer_synthesiser)
builder.add_node("self_reflector",       self_reflector)
builder.add_node("query_rewriter",       query_rewriter)

# ── Wire edges ────────────────────────────────────────────────────────────────
builder.add_edge(START,                  "query_planner")
builder.add_edge("query_planner",        "faiss_retriever")
builder.add_edge("faiss_retriever",      "iterative_check")

# After relevance check: relevant → answer directly, not relevant → ReAct
builder.add_conditional_edges(
    "iterative_check",
    should_call_react,
    path_map=["cot_answerer", "react_router"],
)

# ReAct always feeds into CoT answering after fetching external docs
builder.add_edge("react_router",         "cot_answerer")
builder.add_edge("cot_answerer",         "answer_synthesiser")
builder.add_edge("answer_synthesiser",   "self_reflector")

# Self-reflection loop: approved → END, needs revision → rewrite → re-plan
builder.add_conditional_edges(
    "self_reflector",
    should_continue,
    path_map=["query_rewriter", END],
)

# After rewriting, go back to query_planner to re-decompose the new question
builder.add_edge("query_rewriter",       "query_planner")

# ── Compile ──────────────────────────────────────────────────────────────────
graph = builder.compile()
logger.info("LangGraph compiled successfully")

# ── Cell health-check ────────────────────────────────────────────────────────
assert graph is not None, "❌ Graph failed to compile"
print("✅ Cell 7 passed — graph compiled.")
graph   # renders the visual diagram in the notebook

---
## Cell 8 — Run the Agent

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 │ Run the Agent
#
# Modify USER_QUERY to test different questions.
# All intermediate state is accessible in `result` for debugging.
# ─────────────────────────────────────────────────────────────────────────────

USER_QUERY = "What were the key financial highlights in Q3 2025?"

logger.info(f"{'='*60}")
logger.info(f"Starting Agentic RAG | Question: '{USER_QUERY}'")
logger.info(f"{'='*60}")

init_state = AgenticRAGState(
    question=USER_QUERY,
    original_question=USER_QUERY,
)

result = graph.invoke(init_state)

logger.info(f"Agent finished | Attempts: {result['attempts']} | Source: {result['retrieval_source']}")

# ── Display results ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("🗂️  SUB-QUERIES PLANNED:")
for i, sq in enumerate(result["sub_queries"], 1):
    print(f"   {i}. {sq}")

print("\n" + "="*60)
print(f"📡 RETRIEVAL SOURCE: {result['retrieval_source'].upper()}")

print("\n" + "="*60)
print("🧠 FINAL ANSWER:")
print(result["final_answer"])

print("\n" + "="*60)
print("🔍 REFLECTION:")
print(result["reflection"])

print("\n" + "="*60)
print(f"🔄 TOTAL ATTEMPTS: {result['attempts']}")